In [1]:
!pip -q install duckdb huggingface_hub

from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute("SET VARIABLE hf_token = ?", [token])

con.execute("""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN getvariable('hf_token')
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Connected to March 2026 warehouse data.")

Connected to March 2026 warehouse data.


# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Malang43/flyrank-ml-internship-Malang43/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Lane: Refresh / Content Opportunity Scoring

Before defining my baseline rule, I check two observable signals that the rule depends on: search position versus CTR, and impression volume. The first is connected to CTR-review logic, while the second checks whether a page has enough search exposure to justify prioritizing human review.

Baseline rule: Prioritize pages that have meaningful search visibility, rank within the first 20 search positions, and show relatively weak CTR. These pages may deserve a human review of title/meta information and search-intent alignment.

Reason code: visible_low_ctr

Action label: review_title_meta_intent

This is a decision-support rule only. A high score means review the page first; it does not prove that changing the page will improve performance.

In [2]:
signal_position = con.sql(f"""
SELECT
    CASE
        WHEN gsc_avg_position <= 3 THEN '1-3'
        WHEN gsc_avg_position <= 10 THEN '4-10'
        WHEN gsc_avg_position <= 20 THEN '11-20'
        ELSE '20+'
    END AS position_bucket,

    COUNT(*) AS n,

    ROUND(
        100.0 * SUM(gsc_clicks) /
        NULLIF(SUM(gsc_impressions), 0),
        3
    ) AS ctr_pct

FROM {MAR}

WHERE gsc_data_available IS TRUE
  AND gsc_impressions > 0
  AND gsc_avg_position > 0

GROUP BY 1

ORDER BY
    CASE
        WHEN position_bucket = '1-3' THEN 1
        WHEN position_bucket = '4-10' THEN 2
        WHEN position_bucket = '11-20' THEN 3
        ELSE 4
    END
""").df()

signal_position

,position_bucket,n,ctr_pct
0,1-3,564173,0.381
1,4-10,1456122,0.323
2,11-20,519223,0.315
3,20+,908354,0.131


CTR-vs-position verdict: CONFIRMED. CTR changes substantially across position buckets, so weak CTR should be judged together with search position rather than by CTR alone.

In [3]:
signal_volume = con.sql(f"""
SELECT
    CASE
        WHEN gsc_impressions < 10 THEN '<10'
        WHEN gsc_impressions < 100 THEN '10-99'
        WHEN gsc_impressions < 500 THEN '100-499'
        ELSE '500+'
    END AS impression_bucket,

    COUNT(*) AS n,

    ROUND(AVG(gsc_clicks), 2) AS mean_clicks,

    ROUND(AVG(gsc_avg_position), 2) AS mean_position

FROM {MAR}

WHERE gsc_data_available IS TRUE

GROUP BY 1

ORDER BY
    CASE
        WHEN impression_bucket = '<10' THEN 1
        WHEN impression_bucket = '10-99' THEN 2
        WHEN impression_bucket = '100-499' THEN 3
        ELSE 4
    END
""").df()

signal_volume

,impression_bucket,n,mean_clicks,mean_position
0,<10,1463532,0.01,20.52
1,10-99,1508921,0.10,13.30
2,100-499,537157,0.65,10.94
3,500+,101451,2.94,11.69


Volume verdict: CONFIRMED. Higher-impression pages represent greater observed search exposure, so using a minimum-volume condition helps prevent the baseline from prioritizing very low-volume noise.

In [4]:
baseline_df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,

    100.0 * SUM(gsc_clicks) /
        NULLIF(SUM(gsc_impressions), 0) AS ctr_pct,

    SUM(gsc_sum_position) /
        NULLIF(SUM(gsc_impressions), 0) AS avg_position

FROM {MAR}

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id

HAVING SUM(gsc_impressions) > 0
""").df()

print("Page-level rows:", len(baseline_df))
display(baseline_df.head())

Page-level rows: 176738


,client_hash_id,content_hash_id,impressions,clicks,ctr_pct,avg_position
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,0.000000,4.311688
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,10849.0,22.0,0.202784,8.049866
2,client_62f4a7e64f5e0096,content_e689bc511192751a,61.0,0.0,0.000000,5.885246
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,705.0,1.0,0.141844,5.863830
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,50.0,0.0,0.000000,14.360000


In [5]:
eligible = (
    (baseline_df["impressions"] >= 100) &
    (baseline_df["avg_position"] > 0) &
    (baseline_df["avg_position"] <= 20)
)

# Transparent hand-written score.
# Higher impressions increase priority.
# Lower CTR increases priority.
baseline_df["baseline_score"] = np.where(
    eligible,
    np.log1p(baseline_df["impressions"]) *
    (1 / (1 + baseline_df["ctr_pct"])),
    0
)

baseline_df["reason_code"] = np.where(
    eligible,
    "visible_low_ctr",
    "not_selected"
)

baseline_df["action"] = np.where(
    eligible,
    "review_title_meta_intent",
    "monitor"
)

queue = (
    baseline_df
    .sort_values(
        ["baseline_score", "impressions"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

queue["rank"] = np.arange(1, len(queue) + 1)

display(
    queue[
        [
            "rank",
            "content_hash_id",
            "impressions",
            "clicks",
            "ctr_pct",
            "avg_position",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ].head(20)
)

,rank,content_hash_id,impressions,clicks,ctr_pct,avg_position,baseline_score,reason_code,action
0,1,content_44f34c0a90047651,212404.0,24.0,0.011299,0.665877,12.129200,visible_low_ctr,review_title_meta_intent
1,2,content_8e1334d6356668e3,134984.0,1.0,0.000741,2.693038,11.804174,visible_low_ctr,review_title_meta_intent
2,3,content_fec55986a1868d62,124075.0,1.0,0.000806,0.308426,11.719204,visible_low_ctr,review_title_meta_intent
3,4,content_34a70fea29d15f24,143019.0,43.0,0.030066,3.166132,11.524252,visible_low_ctr,review_title_meta_intent
4,5,content_f6116743b00afc2d,107584.0,15.0,0.013943,9.735658,11.426718,visible_low_ctr,review_title_meta_intent
5,6,content_cd3d932d4e1c8db0,89332.0,4.0,0.004478,7.831807,11.349308,visible_low_ctr,review_title_meta_intent
6,7,content_9c057b66c30a3abb,83834.0,1.0,0.001193,0.116003,11.323099,visible_low_ctr,review_title_meta_intent
7,8,content_046fc480045b88f5,83788.0,6.0,0.007161,7.208276,11.255457,visible_low_ctr,review_title_meta_intent
8,9,content_9540d884af3e41fd,82376.0,11.0,0.013353,8.005184,11.169905,visible_low_ctr,review_title_meta_intent
9,10,content_425715547c6a3ea8,71513.0,3.0,0.004195,6.983444,11.130954,visible_low_ctr,review_title_meta_intent


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "baseline_action_score.csv"

queue.to_csv(output_file, index=False)

print("Saved:", output_file)
print("Queue rows:", len(queue))

Saved: ../outputs/baseline_action_score.csv
Queue rows: 176738


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
top20 = queue.head(20).copy()

display(
    top20[
        [
            "rank",
            "content_hash_id",
            "impressions",
            "ctr_pct",
            "avg_position",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ]
)

,rank,content_hash_id,impressions,ctr_pct,avg_position,baseline_score,reason_code,action
0,1,content_44f34c0a90047651,212404.0,0.011299,0.665877,12.129200,visible_low_ctr,review_title_meta_intent
1,2,content_8e1334d6356668e3,134984.0,0.000741,2.693038,11.804174,visible_low_ctr,review_title_meta_intent
2,3,content_fec55986a1868d62,124075.0,0.000806,0.308426,11.719204,visible_low_ctr,review_title_meta_intent
3,4,content_34a70fea29d15f24,143019.0,0.030066,3.166132,11.524252,visible_low_ctr,review_title_meta_intent
4,5,content_f6116743b00afc2d,107584.0,0.013943,9.735658,11.426718,visible_low_ctr,review_title_meta_intent
5,6,content_cd3d932d4e1c8db0,89332.0,0.004478,7.831807,11.349308,visible_low_ctr,review_title_meta_intent
6,7,content_9c057b66c30a3abb,83834.0,0.001193,0.116003,11.323099,visible_low_ctr,review_title_meta_intent
7,8,content_046fc480045b88f5,83788.0,0.007161,7.208276,11.255457,visible_low_ctr,review_title_meta_intent
8,9,content_9540d884af3e41fd,82376.0,0.013353,8.005184,11.169905,visible_low_ctr,review_title_meta_intent
9,10,content_425715547c6a3ea8,71513.0,0.004195,6.983444,11.130954,visible_low_ctr,review_title_meta_intent


In [8]:
for _, r in top20.iterrows():

    if r["impressions"] >= 1000 and r["avg_position"] <= 10:
        confidence = "high"
    elif r["impressions"] >= 500:
        confidence = "medium"
    else:
        confidence = "low"

    print(
        f"Rank {int(r['rank'])}: "
        f"Action = review title/meta and intent | "
        f"Reason = {r['reason_code']} | "
        f"Confidence = {confidence} | "
        f"Impressions = {r['impressions']:.0f}, "
        f"CTR = {r['ctr_pct']:.2f}%, "
        f"Position = {r['avg_position']:.2f} | "
        f"Could be wrong if low CTR is normal for the query mix, "
        f"SERP layout, seasonality, or short-term noise."
    )

Rank 1: Action = review title/meta and intent | Reason = visible_low_ctr | Confidence = high | Impressions = 212404, CTR = 0.01%, Position = 0.67 | Could be wrong if low CTR is normal for the query mix, SERP layout, seasonality, or short-term noise.
Rank 2: Action = review title/meta and intent | Reason = visible_low_ctr | Confidence = high | Impressions = 134984, CTR = 0.00%, Position = 2.69 | Could be wrong if low CTR is normal for the query mix, SERP layout, seasonality, or short-term noise.
Rank 3: Action = review title/meta and intent | Reason = visible_low_ctr | Confidence = high | Impressions = 124075, CTR = 0.00%, Position = 0.31 | Could be wrong if low CTR is normal for the query mix, SERP layout, seasonality, or short-term noise.
Rank 4: Action = review title/meta and intent | Reason = visible_low_ctr | Confidence = high | Impressions = 143019, CTR = 0.03%, Position = 3.17 | Could be wrong if low CTR is normal for the query mix, SERP layout, seasonality, or short-term noise.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: Rank 14 is a weaker recommendation because its average position is 15.82, so its lower CTR may partly be expected from weaker search position rather than a title/meta problem. Rank 20 also deserves caution because its CTR (0.14%) is higher than many other top-ranked candidates; its very high impression volume is driving much of its priority. These cases show why the baseline is a review queue, not an automatic content-change decision.

Leakage check: This baseline uses only March 2026 information available at the decision moment: impressions, clicks, CTR, and search position. It does not use April outcomes, future-window measurements, future_decline_proxy, target-derived fields, product scores, or decision flags. Therefore, the baseline does not intentionally use future or label-derived information.

In [9]:
baseline_inputs = [
    "impressions",
    "clicks",
    "ctr_pct",
    "avg_position"
]

forbidden_inputs = [
    "future_decline_proxy",
    "april_impressions",
    "label_copy_leak",
    "health_score",
    "priority_score",
    "action_type"
]

print("Baseline inputs:")
for x in baseline_inputs:
    print("  OK:", x)

print("\nForbidden / unused inputs:")
for x in forbidden_inputs:
    print("  EXCLUDED:", x)

Baseline inputs:
  OK: impressions
  OK: clicks
  OK: ctr_pct
  OK: avg_position

Forbidden / unused inputs:
  EXCLUDED: future_decline_proxy
  EXCLUDED: april_impressions
  EXCLUDED: label_copy_leak
  EXCLUDED: health_score
  EXCLUDED: priority_score
  EXCLUDED: action_type


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.